# Speaker Verification — ResNet34 + Triplet Loss

**What changed vs the Contrastive Loss baseline:**
- **Loss function:** `TripletMarginLoss` instead of `ContrastiveLoss`
- **Dataset:** `TripletDataset` — each sample is a **(Anchor, Positive, Negative)** triplet instead of a pair
- **Training signal:** The model is pushed to make `d(anchor, positive) < d(anchor, negative)` by at least `margin`

---
## 1. Dataset Setup

In [ ]:
import os
import pandas as pd
import numpy as np

DATASET_ROOT  = "/kaggle/input/datasets/tahmidulislamomi/ml-project-openslr-dataset"
BASE_AUDIO_DIR = os.path.join(DATASET_ROOT, "train-clean-100", "train-clean-100")
CSV_PATH      = os.path.join(DATASET_ROOT, "train_pairs.csv")

print("BASE_AUDIO_DIR:", BASE_AUDIO_DIR)
print("CSV_PATH:",       CSV_PATH)

## 2. Load Training Data

Load the CSV, resolve absolute paths, and extract speaker IDs from the file paths.

In [ ]:
df = pd.read_csv(CSV_PATH)

def to_absolute_path(rel_path):
    cleaned = rel_path.replace("train-clean-100/", "", 1)
    return os.path.join(BASE_AUDIO_DIR, cleaned)

def get_speaker_id(rel_path):
    # e.g. "train-clean-100/311/124404/311-124404-0059.flac" → "311"
    return rel_path.split("/")[1]

df["path1_abs"]   = df["audio_path_1"].apply(to_absolute_path)
df["path2_abs"]   = df["audio_path_2"].apply(to_absolute_path)
df["speaker1_id"] = df["audio_path_1"].apply(get_speaker_id)
df["speaker2_id"] = df["audio_path_2"].apply(get_speaker_id)

print("Total rows:", len(df))
print("Unique speakers:", df["speaker1_id"].nunique())
print(df.head())

## 3. Audio Transforms & Standardization

Random crop or zero-pad to exactly 3 seconds, then extract Log-Mel Spectrograms.

In [ ]:
import torch
import numpy as np
import torchaudio.transforms as T

TARGET_SR     = 16000
TARGET_LENGTH = TARGET_SR * 3  # 48000 samples

def crop_or_pad(audio):
    length = len(audio)
    if length > TARGET_LENGTH:
        start = np.random.randint(0, length - TARGET_LENGTH)
        audio = audio[start:start + TARGET_LENGTH]
    elif length < TARGET_LENGTH:
        audio = np.pad(audio, (0, TARGET_LENGTH - length), mode='constant')
    return audio

mel_transform  = T.MelSpectrogram(sample_rate=16000, n_fft=400, hop_length=160, n_mels=80)
amplitude_to_db = T.AmplitudeToDB()

## 4. Triplet Dataset

For each item, randomly pick:
- **Anchor** — any audio file
- **Positive** — a *different* file from the *same* speaker
- **Negative** — a file from a *different* speaker

Speaker lookup is built from the CSV so no extra files are needed.

In [ ]:
import soundfile as sf
from collections import defaultdict
from torch.utils.data import Dataset

class TripletDataset(Dataset):
    def __init__(self, dataframe, mel_transform, amplitude_to_db):
        self.mel_transform  = mel_transform
        self.amplitude_to_db = amplitude_to_db

        # Build speaker → [absolute paths] lookup
        self.speaker_to_paths = defaultdict(list)
        for _, row in dataframe.iterrows():
            self.speaker_to_paths[row["speaker1_id"]].append(row["path1_abs"])
            self.speaker_to_paths[row["speaker2_id"]].append(row["path2_abs"])

        # Store (path, speaker_id) pairs for sampling anchors
        self.samples = []
        for spk, paths in self.speaker_to_paths.items():
            for p in paths:
                self.samples.append((p, spk))

        self.all_speakers = list(self.speaker_to_paths.keys())

    def __len__(self):
        return len(self.samples)

    def load_audio(self, path):
        audio, sr = sf.read(path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        audio = crop_or_pad(audio)
        audio = torch.tensor(audio).float().unsqueeze(0)
        mel   = self.mel_transform(audio)
        return self.amplitude_to_db(mel)

    def __getitem__(self, idx):
        anchor_path, anchor_spk = self.samples[idx]

        # Positive: same speaker, different file
        pos_candidates = [p for p in self.speaker_to_paths[anchor_spk] if p != anchor_path]
        if not pos_candidates:
            pos_candidates = self.speaker_to_paths[anchor_spk]
        pos_path = pos_candidates[np.random.randint(len(pos_candidates))]

        # Negative: different speaker
        neg_spk = anchor_spk
        while neg_spk == anchor_spk:
            neg_spk = self.all_speakers[np.random.randint(len(self.all_speakers))]
        neg_paths = self.speaker_to_paths[neg_spk]
        neg_path  = neg_paths[np.random.randint(len(neg_paths))]

        return (
            self.load_audio(anchor_path),
            self.load_audio(pos_path),
            self.load_audio(neg_path),
        )

## 5. Initialize Training Dataset & DataLoader

In [ ]:
from torch.utils.data import DataLoader

dataset    = TripletDataset(df, mel_transform, amplitude_to_db)
print("Dataset size:", len(dataset))

dataloader = DataLoader(
    dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True
)
print("DataLoader ready")

## 6. Model Architecture — ResNetSpeaker (ResNet34)

Exact same model as the Contrastive baseline — only the loss function changes.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class ResNetSpeaker(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()
        self.backbone = models.resnet34(weights=None)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.embedding = nn.Linear(in_features, embedding_dim)

    def forward(self, x):
        features  = self.backbone(x)
        embedding = self.embedding(features)
        return F.normalize(embedding, p=2, dim=1)

## 7. Initialize Model on Device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = ResNetSpeaker(embedding_dim=256).to(device)
print("Using device:", device)
print(model)

## 8. Triplet Margin Loss

`TripletMarginLoss` pushes the model to satisfy:

`d(anchor, positive) + margin < d(anchor, negative)`

- **margin = 0.3** — a widely used default for speaker verification with L2-normalized embeddings
- If already satisfied, loss = 0 for that triplet (no gradient — the model is "hard enough")

In [ ]:
criterion = nn.TripletMarginLoss(margin=0.3, p=2)

## 9. Training Configuration

In [ ]:
model     = ResNetSpeaker(256).to(device)
criterion = nn.TripletMarginLoss(margin=0.3, p=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 10. Training Loop

Trains for **30 epochs**.
- **Triplet accuracy** = fraction of triplets where `d(anchor, pos) < d(anchor, neg)` (model did the right thing)
- Auto-saves checkpoint after every epoch

In [ ]:
from tqdm import tqdm

num_epochs  = 30
print_every = 50

loss_history = []
acc_history  = []

for epoch in range(num_epochs):
    model.train()
    total_loss    = 0
    total_correct = 0
    total_samples = 0

    progress_bar = tqdm(enumerate(dataloader),
                        total=len(dataloader),
                        desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, (anchor, positive, negative) in progress_bar:
        anchor   = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        optimizer.zero_grad()
        emb_a = model(anchor)
        emb_p = model(positive)
        emb_n = model(negative)

        loss = criterion(emb_a, emb_p, emb_n)
        loss.backward()
        optimizer.step()

        # Triplet accuracy: d(a,p) < d(a,n)
        with torch.no_grad():
            d_pos = F.pairwise_distance(emb_a, emb_p)
            d_neg = F.pairwise_distance(emb_a, emb_n)
            total_correct += (d_pos < d_neg).sum().item()
            total_samples += anchor.size(0)

        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

        if batch_idx % print_every == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    avg_acc  = total_correct / total_samples

    loss_history.append(avg_loss)
    acc_history.append(avg_acc)

    print(f"\nEpoch [{epoch+1}/{num_epochs}] - Avg Loss: {avg_loss:.4f} | Triplet Acc: {avg_acc:.4f}\n")

    # Auto-save after every epoch (handles Kaggle 12-hour timeout)
    torch.save({
        "model_state":     model.state_dict(),
        "epoch":           epoch,
        "optimizer_state": optimizer.state_dict(),
        "loss_history":    loss_history,
        "acc_history":     acc_history,
    }, "checkpoint_resnet34_triplet.pth")
    print(f"Checkpoint saved → epoch {epoch+1}")

## 11. Training Graphs

Two side-by-side plots:
- **Left** — Triplet Loss per epoch (should trend downward toward 0)
- **Right** — Triplet Accuracy per epoch (% of triplets where anchor is closer to positive than negative — should trend toward 1.0)

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(loss_history) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("ResNet34 + Triplet Loss — Training Metrics", fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, loss_history, marker='o', color='tomato', linewidth=2)
axes[0].set_title("Triplet Loss per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, acc_history, marker='o', color='steelblue', linewidth=2)
axes[1].set_title("Triplet Accuracy per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (d_pos < d_neg)")
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_metrics_triplet.png", dpi=150)
plt.show()
print("Saved → training_metrics_triplet.png")

## 12. Embedding Distance Distribution

Plots the distribution of:
- 🔵 **Anchor–Positive distances** (same speaker — should be small)
- 🔴 **Anchor–Negative distances** (different speaker — should be large)

A clear separation means the model has learned to differentiate speakers well in embedding space.

In [ ]:
model.eval()
d_pos_all, d_neg_all = [], []

with torch.no_grad():
    for anchor, positive, negative in dataloader:
        emb_a = model(anchor.to(device))
        emb_p = model(positive.to(device))
        emb_n = model(negative.to(device))

        d_pos_all.extend(F.pairwise_distance(emb_a, emb_p).cpu().tolist())
        d_neg_all.extend(F.pairwise_distance(emb_a, emb_n).cpu().tolist())

        if len(d_pos_all) > 5000:
            break

plt.figure(figsize=(8, 5))
plt.hist(d_pos_all, bins=60, alpha=0.6, color='steelblue', label='Anchor–Positive (same speaker)')
plt.hist(d_neg_all, bins=60, alpha=0.6, color='tomato',    label='Anchor–Negative (different speaker)')
plt.xlabel("Euclidean Distance")
plt.ylabel("Count")
plt.title("Embedding Distance Distribution After Training")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("distance_distribution_triplet.png", dpi=150)
plt.show()
print("Saved → distance_distribution_triplet.png")

## 13. Save Final Model

In [ ]:
torch.save({
    "model_state":     model.state_dict(),
    "epoch":           epoch,
    "optimizer_state": optimizer.state_dict(),
    "loss_history":    loss_history,
    "acc_history":     acc_history,
}, "checkpoint_resnet34_triplet.pth")
print("Final model saved to checkpoint_resnet34_triplet.pth")